## Motivation of Vanna-Volga Pricing 
Black & Scholes provides adequate & fast pricing for vanilla or even barrier options, but they are wrong from market quotes, as the underlying assumptions don't take into volatility and rate stochasticity. This modification necessitates a more flexible modelling and calibration framework, taking into account spot-vol and smile dynamics.

More Realistic models assume stochastic rates or volatility, but this is generally hard to calibrate computationally and demands a lot of market date to find suitalbe parameters.

Since traders need to quote prices in a fast paced environment, they have come up with a pratical rule of thumb to calibrate to market quotes, and atleast avoid arbitrage to a certain degree. The idea is to quantify skew, wing vs belly dynamics and backbone ATM vol using straddles, Risk reversals and Butterfly spreads. This method is fast, and produces reasonable estimates of market prices for vanillas as well as first generation exotics.

it's based on adding an analytically derived correction to the B&S price, by constructing a replication portfolio, that neutralizes greeks : Vega, Vanna and Volga (thus the name) producing smile consistent values.

In [15]:
import numpy as np
from math import sqrt, exp, log
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq
from datetime import datetime

In [16]:
import math
from scipy.stats import norm

def fx_option_pricer(flag, S, K, rd, rf, T, impvol):
    
    d1 = (log(S / K) + (rd - rf + 0.5 * impvol**2) * T) / (impvol * sqrt(T))
    d2 = d1 - impvol * sqrt(T)

    if flag.upper() == "C":
        # FX Call option 
        price = S * exp(-rf * T) * norm.cdf(d1) - K * exp(-rd * T) * norm.cdf(d2)
        
    elif flag.upper() == "P":
        # FX Put option (requires -d2 and -d1)
        price = K * exp(-rd * T) * norm.cdf(-d2) - S * exp(-rf * T) * norm.cdf(-d1)
        
    else:
        raise ValueError("Flag must be 'C' or 'P'.")
        
    return price

def fx_implied_vol(target_price, flag, S, K, rd, rf, T):
    def objective_function(vol):
        return fx_option_pricer(flag, S, K, rd, rf, T, vol) - target_price

    try:
        implied_vol = brentq(objective_function, 1e-5, 1.0)
        return implied_vol
    
    except ValueError:
        print("Error: Target price is outside theoretical bounds.")
        return None

### Example Pricing

In [17]:
spot_rate = 1.1391
strike = 1.1400
usd_rate = 0.0366
eur_rate = 0.0244
time_to_mat = 0.5
market_price = 0.035

iv = fx_implied_vol(market_price, "C", spot_rate, strike, usd_rate, eur_rate, time_to_mat)

print(f"Market Price: {market_price}")
print(f"Implied Volatility: {iv:.4%} ({iv:.4f})")

Market Price: 0.035
Implied Volatility: 10.0881% (0.1009)


## Volatility Interpolation

In [18]:
spot_rate = 1.1391
strike = 1.1400
usd_rate = 0.0366
eur_rate = 0.0244
TTM = 0.5
Vol_data = [(1.135, .12), (1.14, 0.08), (1.145, .140)]

FX_instruments = [fx_option_pricer("C", spot_rate, K, usd_rate, eur_rate, TTM,vol) for K,vol in Vol_data]

for i in FX_instruments:
    print(f"Market Price: {i:.4%}")

Market Price: 4.3579%
Market Price: 2.8416%
Market Price: 4.4930%


In [19]:
def fx_vega(flag, S, K, rd, rf, T, impvol, eps=1e-4):
    p_up = fx_option_pricer(flag, S, K, rd, rf, T, impvol + eps)
    p_down = fx_option_pricer(flag, S, K, rd, rf, T, impvol - eps)
    return (p_up - p_down) / (2 * eps)

def fx_vanna(flag, S, K, rd, rf, T, impvol, eps_vol=1e-4, eps_spot=1e-4):
    vega_up = fx_vega(flag, S + eps_spot, K, rd, rf, T, impvol, eps_vol)
    vega_down = fx_vega(flag, S - eps_spot, K, rd, rf, T, impvol, eps_vol)
    return (vega_up - vega_down) / (2 * eps_spot)

def fx_volga(flag, S, K, rd, rf, T, impvol, eps=1e-4):
    vega_up = fx_vega(flag, S, K, rd, rf, T, impvol + eps, eps)
    vega_down = fx_vega(flag, S, K, rd, rf, T, impvol - eps, eps)
    return (vega_up - vega_down) / (2 * eps)

In [ ]:
spot_rate = 1.1391
usd_rate = 0.0366
eur_rate = 0.0244
TTM = 0.5
Vol_data = [(1.135, .12), (1.14, 0.08), (1.145, .140)]

results = []
for K, vol in Vol_data:
    price = fx_option_pricer("C", spot_rate, K, usd_rate, eur_rate, TTM, vol)
    vega  = fx_vega("C", spot_rate, K, usd_rate, eur_rate, TTM, vol)
    vanna = fx_vanna("C", spot_rate, K, usd_rate, eur_rate, TTM, vol)
    volga = fx_volga("C", spot_rate, K, usd_rate, eur_rate, TTM, vol)
    
    results.append({
        "Strike": K,
        "Vol": vol,
        "Price": round(price, 5),
        "Vega": round(vega, 4),
        "Vanna": round(vanna, 4),
        "Volga": round(volga, 4)
    })

# Print out the results
for res in results:
    print(res)

{'Strike': 1.135, 'Vol': 0.12, 'Price': np.float64(0.04358), 'Vega': np.float64(0.3136), 'Vanna': np.float64(-0.2334), 'Volga': np.float64(0.0295)}
{'Strike': 1.14, 'Vol': 0.08, 'Price': np.float64(0.02842), 'Vega': np.float64(0.3151), 'Vanna': np.float64(-0.3207), 'Volga': np.float64(0.0316)}
{'Strike': 1.145, 'Vol': 0.14, 'Price': np.float64(0.04493), 'Vega': np.float64(0.3169), 'Vanna': np.float64(0.1126), 'Volga': np.float64(-0.0053)}
